<h2>A beginner-friendly CNN project for classifying images as cats or dogs. 
then used to train and evaluate a Convolutional Neural Network (CNN).</h2>

In [133]:
#ALl the neccsary import for the Dogs and cat classification
import torch
import torchvision
import torch.nn as nn
import torch.optim as optim
import pandas as pd
import numpy as np
from torchvision.datasets import CIFAR10
from torch.utils.data import DataLoader,TensorDataset
import torchvision.transforms as transforms
from torchvision.datasets import ImageFolder

<h3>Transforming the data Along with setting up training and testing data</h3>

In [134]:
transform=transforms.Compose([
    transforms.Resize((400, 400)),
    transforms.RandomHorizontalFlip(),
    transforms.RandomRotation(10),
    transforms.ColorJitter(
        brightness=0.2,
        contrast=0.2,
        saturation=0.2
    ),
    transforms.ToTensor(),
    transforms.Normalize((0.5,0.5,0.5),(0.5,0.5,0.5))
])
#Here we Specify the path of the folder for train and val
trainset = ImageFolder(
    root="./train",
    transform=transform
)
valset = ImageFolder(
    root="./val",
    transform=transform
)

## Calculating the average of the size of images for fc layer

In [135]:
from PIL import Image

widths = []
heights = []

for path, label in trainset.samples:
    img = Image.open(path)
    width, height = img.size

    widths.append(width)
    heights.append(height)

avg_width = sum(widths) / len(widths)
avg_height = sum(heights) / len(heights)

print("Average width:", avg_width)
print("Average height:", avg_height)

Average width: 443.1636363636364
Average height: 381.70545454545453


<h3>Now loading the Train and Val  data into Loader for CNN training</h3>

In [136]:
trainloader=DataLoader(trainset,batch_size=64,shuffle=True)
valloader=DataLoader(valset,batch_size=64,shuffle=True)

## Making the CNN model for training

In [137]:
class CNN(nn.Module):
    def __init__(self):
        super(CNN,self).__init__()
        #Convoulation layer and pooling layer 
        self.conv_layers=nn.Sequential(
            #input layer with 3 channel rgb with 32 feature output
            nn.Conv2d(3,32,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            #first hidden layer with 32 input layer from input layer and 64 output layer
            nn.Conv2d(32,64,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
            #Seocnd hidden layer with 64 input layer from input layer and 128 output layer
            nn.Conv2d(64,128,kernel_size=3,padding=1),
            nn.ReLU(),
            nn.MaxPool2d(2,2),
        )
        #forward layering as the last step
        self.fc_layers=nn.Sequential(
            #using the average values from the above code in our last layer
            nn.Linear(128*50*50,256),
            nn.ReLU(),
            nn.Dropout(0.5),
            #Setting up for the final output 
            nn.Linear(256, 2),
        )
    def forward(self,x):
            x=self.conv_layers(x)
            x=x.view(x.size(0),-1)
            x=self.fc_layers(x)
            return x
        

In [138]:
#Calling the model and setting up loss and criteria functions
model=CNN()
optimizer=optim.Adam(model.parameters())
criterion = nn.CrossEntropyLoss()

## Training the model 


In [139]:
epochs = 20

for epoch in range(epochs):
    epoch_training_loss = 0.0

    for images, labels in trainloader:
        optimizer.zero_grad()

        output = model(images)

        loss = criterion(output, labels)

        loss.backward()
        optimizer.step()

        epoch_training_loss += loss.item()

    print(f"Epoch {epoch+1} & loss {epoch_training_loss / len(trainloader)}")

Epoch 1 & loss 3.902060842514038
Epoch 2 & loss 0.657096266746521
Epoch 3 & loss 0.6466087222099304
Epoch 4 & loss 0.6393265724182129
Epoch 5 & loss 0.6064532041549683
Epoch 6 & loss 0.5884158849716187
Epoch 7 & loss 0.618022084236145
Epoch 8 & loss 0.581744372844696
Epoch 9 & loss 0.5700216054916382
Epoch 10 & loss 0.5959274172782898
Epoch 11 & loss 0.5996246576309204
Epoch 12 & loss 0.5734141707420349
Epoch 13 & loss 0.5579240262508393
Epoch 14 & loss 0.5503786683082581
Epoch 15 & loss 0.5687066912651062
Epoch 16 & loss 0.5677565515041352
Epoch 17 & loss 0.542288088798523
Epoch 18 & loss 0.5137721061706543
Epoch 19 & loss 0.5267195224761962
Epoch 20 & loss 0.5152584612369537


## Testing the model with train set and validation set

In [140]:
model.eval()

# TRAIN ACCURACY
corrected_label = 0
total_label = 0

with torch.no_grad():
    for images, labels in trainloader:
        output = model(images)
        _, predicted = torch.max(output, 1)

        corrected_label += (predicted == labels).sum().item()
        total_label += labels.size(0)

print(f"Train Accuracy = {corrected_label / total_label * 100:.2f}%")


# VALIDATION ACCURACY
corrected_label = 0
total_label = 0

with torch.no_grad():
    for images, labels in valloader:
        output = model(images)
        _, predicted = torch.max(output, 1)

        corrected_label += (predicted == labels).sum().item()
        total_label += labels.size(0)

print(f"Validation Accuracy = {corrected_label / total_label * 100:.2f}%")

Train Accuracy = 74.91%
Validation Accuracy = 61.43%


## Testing with random images

In [141]:
from PIL import Image
from torchvision import transforms
import torch

images = ["cat1.png", "cat-dog.png","bull.png","dog4.png","cat3.png","cat4.png","dog.png","dog1.png"]

transform = transforms.Compose([
    transforms.Resize((400, 400)),
    transforms.ToTensor()
])

model.eval()

for filename in images:
    img = Image.open(filename).convert("RGB")
    img = transform(img).unsqueeze(0)

    with torch.no_grad():
        output = model(img)
        prob = torch.softmax(output, dim=1)
        pred = torch.argmax(prob, dim=1).item()

    print(filename)
    print(pred)
    print("Prediction:", ["cat", "dog"][pred])
    print("Confidence:", prob[0][pred].item() * 100, "%")
    print()

cat1.png
0
Prediction: cat
Confidence: 54.141831398010254 %

cat-dog.png
1
Prediction: dog
Confidence: 90.89689254760742 %

bull.png
0
Prediction: cat
Confidence: 74.97095465660095 %

dog4.png
0
Prediction: cat
Confidence: 73.07355999946594 %

cat3.png
0
Prediction: cat
Confidence: 67.03985929489136 %

cat4.png
0
Prediction: cat
Confidence: 66.21898412704468 %

dog.png
1
Prediction: dog
Confidence: 56.93698525428772 %

dog1.png
1
Prediction: dog
Confidence: 55.77893853187561 %



In [144]:
!git init

Initialized empty Git repository in C:/Coding/apna college/Deep_Learning/.git/
